# M005 – Risk Segment Performance Mart Validation

## Objective

This notebook validates the Risk Segment Performance Mart built from the LendingClub feature store.

The mart aggregates portfolio performance across key borrower risk segments and evaluates how default behaviour changes across underwriting dimensions.

The analysis focuses on:

- Grade-based risk segmentation
- Utilization-based segmentation
- High DTI borrowers
- Thin-file borrowers
- Borrowers with bankruptcy history

This mart provides a portfolio-level view of which borrower segments contribute the most credit risk and is directly reusable in underwriting, portfolio monitoring, collections, and IFRS9 workflows.


In [2]:
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

DB_PATH = (
    Path.cwd()
    / "projects"
    / "P0_Data_Platform"
    / "datasets"
    / "lendingclub"
    / "data"
    / "warehouse"
    / "duckdb"
    / "lendingclub.duckdb"
)

print(DB_PATH)

conn = duckdb.connect(str(DB_PATH))


d:\project_lighthouse\projects\P0_Data_Platform\datasets\lendingclub\data\warehouse\duckdb\lendingclub.duckdb


## Mart Row Count Validation

The mart should contain one row for every risk segment value across all supported segment types.


In [3]:
conn.execute('''
select count(*) as mart_rows
from mart.risk_segment_performance
''').fetchdf()

,mart_rows
0,17


## Findings

The mart contains 17 segment records.

These records represent all supported risk segment categories and their corresponding segment values.

The row count is consistent with the intended mart design and confirms successful aggregation of portfolio risk segments.


## Segment Coverage Validation

The mart should contain all planned segment types.


In [4]:
conn.execute('''
select
    segment_type,
    count(*) as segment_count
from mart.risk_segment_performance
group by segment_type
order by segment_type
''').fetchdf()

,segment_type,segment_count
0,BANKRUPTCY_FLAG,2
1,GRADE,7
2,HIGH_DTI_FLAG,2
3,THIN_FILE_FLAG,2
4,UTILIZATION_BAND,4


## Findings

The mart successfully contains five major segmentation dimensions:

- Grade
- Utilization Band
- High DTI Flag
- Thin File Flag
- Bankruptcy Flag

These dimensions collectively capture borrower quality, leverage, credit history depth, and prior credit distress.


## Schema Review

The mart should contain exposure, borrower quality, pricing, utilization, and performance metrics.


In [5]:
conn.execute('''
describe mart.risk_segment_performance
''').fetchdf()

,column_name,column_type,null,key,default,extra
0,segment_type,VARCHAR,YES,None,None,None
1,segment_value,VARCHAR,YES,None,None,None
2,loan_count,BIGINT,YES,None,None,None
3,total_loan_amount,DOUBLE,YES,None,None,None
4,avg_loan_amount,DOUBLE,YES,None,None,None
5,avg_fico,DOUBLE,YES,None,None,None
6,avg_dti,DOUBLE,YES,None,None,None
7,avg_interest_rate,DOUBLE,YES,None,None,None
8,avg_loan_to_income_ratio,DOUBLE,YES,None,None,None
9,avg_revolving_utilization_ratio,DOUBLE,YES,None,None,None


## Findings

The mart includes:

- Loan counts
- Exposure balances
- Average loan size
- FICO score
- DTI
- Interest rate
- Loan-to-income ratio
- Utilization ratio
- Default count
- Default rate

The schema supports both operational monitoring and strategic risk analysis.


## Highest Risk Segment Analysis

The following output identifies the highest observed default-rate segments.


In [6]:
conn.execute('''
select *
from mart.risk_segment_performance
order by default_rate desc
limit 25
''').fetchdf()

,segment_type,segment_value,loan_count,total_loan_amount,avg_loan_amount,avg_fico,avg_dti,avg_interest_rate,avg_loan_to_income_ratio,avg_revolving_utilization_ratio,default_count,default_rate
0,GRADE,G,12168,2.480324e+08,20383.988741,681.238577,22.432815,28.074255,0.325622,59.283907,4868.0,40.01
1,GRADE,F,41800,7.994102e+08,19124.646531,682.383744,21.676804,25.454091,0.329392,60.039823,15225.0,36.42
2,GRADE,E,135639,2.367318e+09,17453.078392,684.421169,21.549435,21.829653,0.764530,59.132823,38365.0,28.28
3,GRADE,D,324424,5.097344e+09,15711.983007,685.956383,20.929692,18.143067,0.697766,57.443781,66025.0,20.35
4,HIGH_DTI_FLAG,1,246031,3.802132e+09,15453.872683,699.853768,37.730138,15.103867,0.385741,55.222914,40599.0,16.50
5,UTILIZATION_BAND,VERY_HIGH,312800,5.015488e+09,16034.168958,685.265473,20.372044,14.935848,0.405475,88.915969,48351.0,15.46
6,BANKRUPTCY_FLAG,1,271920,3.515323e+09,12927.782436,682.665876,18.429021,13.833510,0.561133,45.262726,41147.0,15.13
7,GRADE,C,650053,9.775551e+09,15038.083318,691.274291,19.552054,14.143689,0.551594,54.435478,93359.0,14.36
8,UTILIZATION_BAND,HIGH,829091,1.313491e+10,15842.542013,688.941482,20.038897,13.867256,0.594111,63.989727,119058.0,14.36
9,THIN_FILE_FLAG,0,2228216,3.370967e+10,15128.547939,700.485828,18.954344,13.078737,0.560017,50.291443,287119.0,12.89


## Findings

The riskiest segments in the portfolio are concentrated within lower LendingClub grades.

Key observations:

- Grade G default rate: 40.01%
- Grade F default rate: 36.42%
- Grade E default rate: 28.28%
- Grade D default rate: 20.35%

A clear monotonic relationship exists between grade quality and default risk.

High DTI borrowers, very-high utilization borrowers, and borrowers with prior bankruptcies also exhibit elevated default rates relative to the portfolio average.

These results confirm that the engineered features successfully separate risk across the borrower population.


## Grade Performance Analysis

Grade is the primary LendingClub underwriting segmentation variable.

The following output evaluates performance across all grades.


In [7]:
conn.execute('''
select *
from mart.risk_segment_performance
where segment_type = 'GRADE'
order by segment_value
''').fetchdf()

,segment_type,segment_value,loan_count,total_loan_amount,avg_loan_amount,avg_fico,avg_dti,avg_interest_rate,avg_loan_to_income_ratio,avg_revolving_utilization_ratio,default_count,default_rate
0,GRADE,A,433027,6.323642e+09,14603.343210,730.992703,16.238648,7.084545,0.527162,37.064114,15536.0,3.59
1,GRADE,B,663557,9.404818e+09,14173.338199,701.831470,17.966630,10.675806,0.546628,48.943646,57449.0,8.66
2,GRADE,C,650053,9.775551e+09,15038.083318,691.274291,19.552054,14.143689,0.551594,54.435478,93359.0,14.36
3,GRADE,D,324424,5.097344e+09,15711.983007,685.956383,20.929692,18.143067,0.697766,57.443781,66025.0,20.35
4,GRADE,E,135639,2.367318e+09,17453.078392,684.421169,21.549435,21.829653,0.764530,59.132823,38365.0,28.28
5,GRADE,F,41800,7.994102e+08,19124.646531,682.383744,21.676804,25.454091,0.329392,60.039823,15225.0,36.42
6,GRADE,G,12168,2.480324e+08,20383.988741,681.238577,22.432815,28.074255,0.325622,59.283907,4868.0,40.01


## Findings

Grade performance follows a highly intuitive risk progression.

| Grade | Default Rate |
|---------|---------|
| A | 3.59% |
| B | 8.66% |
| C | 14.36% |
| D | 20.35% |
| E | 28.28% |
| F | 36.42% |
| G | 40.01% |

As borrower quality declines:

- Default rates increase
- Interest rates increase
- Average FICO scores decrease
- Borrower leverage increases

This validates both LendingClub's grading framework and the feature engineering process used within the warehouse.


# Validation Conclusion

The Risk Segment Performance Mart successfully passed all validation controls.

### Controls Passed

✓ Mart created successfully

✓ Segment coverage validation

✓ Schema validation

✓ Risk segment analysis

✓ Grade performance analysis

### Key Business Findings

The strongest risk differentiation within the portfolio is driven by LendingClub grades.

Grade A borrowers default at only 3.59%, while Grade G borrowers default at 40.01%, representing more than an eleven-fold increase in credit risk.

Additional risk signals are visible among:

- High DTI borrowers
- Very-high utilization borrowers
- Borrowers with bankruptcy history

### Business Value

The mart provides a reusable framework for:

- Underwriting policy review
- Portfolio monitoring
- Collections prioritization
- Credit strategy development
- IFRS9 segmentation analysis

### Validation Outcome

M005 – Risk Segment Performance Mart is complete and validated.

The Mart Layer of Project Lighthouse can now be considered complete.


In [8]:
conn.close()